# Timestamps for each subject

In [76]:
import pandas as pd

# Create the data as a list of dictionaries
data = [
    {"Subject": 1, "Start time": "2024-11-19 18:08:56", "End time": "2024-11-19 18:11:56"},
    {"Subject": 2, "Start time": "2024-11-19 18:16:30", "End time": "2024-11-19 18:21:30"},
    {"Subject": 3, "Start time": "2024-11-19 18:23:33", "End time": "2024-11-19 18:26:33"},
    {"Subject": 4, "Start time": "2024-11-19 18:29:18", "End time": "2024-11-19 18:32:18"},
    {"Subject": 5, "Start time": "2024-11-19 18:34:40", "End time": "2024-11-19 18:37:40"},
    {"Subject": 6, "Start time": "2024-11-19 18:39:30", "End time": "2024-11-19 18:42:30"},
    {"Subject": 7, "Start time": "2024-11-19 18:46:00", "End time": "2024-11-19 18:49:00"}
]

# Convert the list into a pandas DataFrame
df_time = pd.DataFrame(data)

# Convert 'Start time' and 'End time' columns to pandas datetime objects
df_time['Start time'] = pd.to_datetime(df_time['Start time'])
df_time['End time'] = pd.to_datetime(df_time['End time'])

# Print the resulting DataFrame
print(df_time)


   Subject          Start time            End time
0        1 2024-11-19 18:08:56 2024-11-19 18:11:56
1        2 2024-11-19 18:16:30 2024-11-19 18:21:30
2        3 2024-11-19 18:23:33 2024-11-19 18:26:33
3        4 2024-11-19 18:29:18 2024-11-19 18:32:18
4        5 2024-11-19 18:34:40 2024-11-19 18:37:40
5        6 2024-11-19 18:39:30 2024-11-19 18:42:30
6        7 2024-11-19 18:46:00 2024-11-19 18:49:00


In [77]:
# Assuming the times are in Eastern Standard Time (EST)
df_time['Start time'] = df_time['Start time'].dt.tz_localize('US/Eastern', ambiguous='NaT').dt.tz_convert('UTC')
df_time['End time'] = df_time['End time'].dt.tz_localize('US/Eastern', ambiguous='NaT').dt.tz_convert('UTC')

# Print the resulting DataFrame
print(df_time)

   Subject                Start time                  End time
0        1 2024-11-19 23:08:56+00:00 2024-11-19 23:11:56+00:00
1        2 2024-11-19 23:16:30+00:00 2024-11-19 23:21:30+00:00
2        3 2024-11-19 23:23:33+00:00 2024-11-19 23:26:33+00:00
3        4 2024-11-19 23:29:18+00:00 2024-11-19 23:32:18+00:00
4        5 2024-11-19 23:34:40+00:00 2024-11-19 23:37:40+00:00
5        6 2024-11-19 23:39:30+00:00 2024-11-19 23:42:30+00:00
6        7 2024-11-19 23:46:00+00:00 2024-11-19 23:49:00+00:00


# Masimo SPO2 data

In [78]:
import os
import pandas as pd
from datetime import datetime
import pytz

def process_csv(input_directory, output_directory):
    # List all CSV files in the input directory
    csv_files = [f for f in os.listdir(input_directory) if f.endswith('.csv')]
    
    # Initialize an empty DataFrame for merging all CSVs
    merged_df = pd.DataFrame()
    
    # Define the column headers that will be manually inserted
    column_headers = ["Session", "Index", "Timestamp", "Date", "Time", "O2 Saturation", "Pulse Rate", "Perfusion Index"]
    
    # Loop through all the CSV files in the directory
    for file in csv_files:
        file_path = os.path.join(input_directory, file)
        
        # Read the CSV into a DataFrame, skipping lines that are either empty or contain the header line
        df = pd.read_csv(file_path, skip_blank_lines=True)
        
        # Skip rows that are exactly the same as the header line
        df = df[df.iloc[:, 0] != "Session"]
        
        # Drop any rows that are completely empty (if any)
        df.dropna(how='all', inplace=True)
        
        # Merge the current dataframe into the master dataframe
        merged_df = pd.concat([merged_df, df], ignore_index=True)
    
    # Manually set the correct column headers
    merged_df.columns = column_headers
    
    # Combine "Date" and "Time" to form a new "Timestamp" column
    # Remove the timezone part from the "Time" column (e.g., "Eastern Standard Time")
    merged_df["Time"] = merged_df["Time"].str.replace(r' \w+ \w+ Time$', '', regex=True)
    merged_df["Timestamp"] = merged_df["Date"] + " " + merged_df["Time"]
    
    # Convert the "Timestamp" column to datetime with the format "%A, %B %d, %Y %I:%M:%S %p"
    merged_df["Timestamp"] = pd.to_datetime(merged_df["Timestamp"], format="%A, %B %d, %Y %I:%M:%S %p", errors='coerce')
    
    # Check if any timestamps failed to parse
    if merged_df["Timestamp"].isna().any():
        print(f"Some Timestamps could not be parsed: {merged_df[merged_df['Timestamp'].isna()]}")
    
    # Convert "Timestamp" to America/New_York timezone and then to UTC
    merged_df["Timestamp"] = merged_df["Timestamp"].dt.tz_localize("America/New_York", ambiguous='NaT')
    merged_df["Timestamp"] = merged_df["Timestamp"].dt.tz_convert("UTC")
    
    # Save the final dataframe to a CSV file in the specified output directory
    output_path = os.path.join(output_directory, "merged_spo2.csv")
    merged_df.to_csv(output_path, index=False)
    
    print(f"CSV file saved at {output_path}")

    # Return the final DataFrame
    return merged_df

In [79]:
# Specify the directories to use
input_path = "/Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/SPO2"
output_path = "/Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/SPO2/preprocessed/"


# Process the SpO2 data and merge it into a single DataFrame
df_data = process_csv(input_path, output_path)


df_data.head()


CSV file saved at /Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/SPO2/preprocessed/merged_spo2.csv


,Session,Index,Timestamp,Date,Time,O2 Saturation,Pulse Rate,Perfusion Index
0,1,0,2024-11-19 23:15:45+00:00,"Tuesday, November 19, 2024",6:15:45 PM,--,--,--
1,1,1,2024-11-19 23:15:46+00:00,"Tuesday, November 19, 2024",6:15:46 PM,--,--,--
2,1,2,2024-11-19 23:15:47+00:00,"Tuesday, November 19, 2024",6:15:47 PM,--,--,--
3,1,3,2024-11-19 23:15:48+00:00,"Tuesday, November 19, 2024",6:15:48 PM,--,--,--
4,1,4,2024-11-19 23:15:49+00:00,"Tuesday, November 19, 2024",6:15:49 PM,--,60,1.3


In [80]:
# # Specify the output file path
# output_file_path = "/Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/SPO2/preprocessed/merged_spo2.csv"

# # Output the final DataFrame to a CSV file
# df_data.to_csv(output_file_path, index=False)

# logging.info(f"Final preprocessed data has been saved to {output_file_path}")

In [81]:
import pandas as pd

# Load the combined CSV into a DataFrame
input_csv_path = "/Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/SPO2/preprocessed/merged_spo2.csv"
df_spo2 = pd.read_csv(input_csv_path)

# Convert the 'Timestamp' column to datetime
df_spo2['Timestamp'] = pd.to_datetime(df_spo2['Timestamp'], errors='coerce')

# Check if the 'Timestamp' column is already timezone-aware
if df_spo2['Timestamp'].dt.tz is None:
    # If not timezone-aware, localize to UTC
    df_spo2['Timestamp'] = df_spo2['Timestamp'].dt.tz_localize('UTC', ambiguous='NaT')
else:
    # If already timezone-aware, convert to UTC
    df_spo2['Timestamp'] = df_spo2['Timestamp'].dt.tz_convert('UTC')

# Initialize a list to store filtered rows
filtered_rows = []

# Loop through each row in df_time to filter based on the time ranges and assign subject numbers
for _, row in df_time.iterrows():
    # Select the start and end times for the subject
    start_time = row['Start time']
    end_time = row['End time']
    subject_number = row['Subject']  # Get the subject number
    
    # Filter the rows in df_spo2 where the Timestamp is between the start and end times
    filtered_df = df_spo2[(df_spo2['Timestamp'] >= start_time) & (df_spo2['Timestamp'] <= end_time)]
    
    # Make a copy of the filtered DataFrame to avoid SettingWithCopyWarning
    filtered_df = filtered_df.copy()
    
    # Assign the subject number to the filtered rows in the copy
    filtered_df.loc[:, 'Subject'] = subject_number  # This assigns the subject number safely
    
    # Append the filtered rows to the list
    filtered_rows.append(filtered_df)

# Concatenate all the filtered rows into one DataFrame
final_df = pd.concat(filtered_rows, ignore_index=True)

# Save the final DataFrame to a new CSV file
output_csv_path = "/Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/SPO2/preprocessed/parsed_spo2.csv"
final_df.to_csv(output_csv_path, index=False)

print(f"Filtered CSV file saved at {output_csv_path}")


Filtered CSV file saved at /Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/SPO2/preprocessed/parsed_spo2.csv


# Bass Wearable raw PPG dataset

In [88]:
import os
import pandas as pd

def preprocess_ppg_data(input_directory, output_directory, df_time):
    # List all CSV files in the input directory
    csv_files = [f for f in os.listdir(input_directory) if f.endswith('.csv')]
    
    # Initialize an empty DataFrame for merging all CSVs
    merged_df = pd.DataFrame()
    
    # Loop through all the CSV files in the directory
    for file in csv_files:
        file_path = os.path.join(input_directory, file)
        
        # Read the CSV into a DataFrame
        df = pd.read_csv(file_path)
        
        # Merge the current dataframe into the master dataframe
        merged_df = pd.concat([merged_df, df], ignore_index=True)
    
    # Convert 'isodate' column to datetime
    merged_df['Timestamp'] = pd.to_datetime(merged_df['isodate'], errors='coerce')
    
    # Check if the 'Timestamp' column is already timezone-aware
    if merged_df['Timestamp'].dt.tz is None:
        # If not timezone-aware, localize to UTC
        merged_df['Timestamp'] = merged_df['Timestamp'].dt.tz_localize('UTC', ambiguous='NaT')
    else:
        # If already timezone-aware, convert to UTC
        merged_df['Timestamp'] = merged_df['Timestamp'].dt.tz_convert('UTC')
    
    # Drop the original 'isodate' column as it's no longer needed
    merged_df.drop(columns=['isodate'], inplace=True)

    # Drop the unnecessary columns
    merged_df.drop(columns=['waypoints', 'battery', 'air_temp', 'thermistor', 'press', 'humid'], inplace=True)

    # Reorder columns so 'Timestamp' is the first column
    columns = ['Timestamp'] + [col for col in merged_df.columns if col != 'Timestamp']
    merged_df = merged_df[columns]

    # Create a list to store filtered rows
    filtered_rows = []

    # Loop through each row in df_time to filter based on the time ranges and assign subject numbers
    for _, row in df_time.iterrows():
        # Select the start and end times for the subject
        start_time = row['Start time']
        end_time = row['End time']
        subject_number = row['Subject']  # Get the subject number
        
        # Filter the rows in merged_df where the Timestamp is between the start and end times
        filtered_df = merged_df[(merged_df['Timestamp'] >= start_time) & (merged_df['Timestamp'] <= end_time)]
        
        # Make a copy of the filtered DataFrame to avoid SettingWithCopyWarning
        filtered_df = filtered_df.copy()
        
        # Assign the subject number to the filtered rows in the copy
        filtered_df.loc[:, 'Subject'] = subject_number  # This assigns the subject number safely
        
        # Append the filtered rows to the list
        filtered_rows.append(filtered_df)

    # Concatenate all the filtered rows into one DataFrame
    final_df = pd.concat(filtered_rows, ignore_index=True)

    # Save the final DataFrame to a new CSV file in the specified output directory
    output_csv_path = os.path.join(output_directory, "merged_ppg_data.csv")
    final_df.to_csv(output_csv_path, index=False)

    print(f"Filtered and merged CSV file saved at {output_csv_path}")

    # Return the final DataFrame
    return final_df

In [89]:

input_dir = "/Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/BassWearable"
output_dir = "/Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/BassWearable/preprocessed"

# Assuming df_time is already loaded
df_bass = preprocess_ppg_data(input_dir, output_dir, df_time)


df_bass.head()


Filtered and merged CSV file saved at /Users/sjtok/bc_infection/wearable-data-science/data/11_19_2024/BassWearable/preprocessed/merged_ppg_data.csv


,Timestamp,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,mag_x,mag_y,mag_z,ppg_r,ppg_b,ppg_ir,ppg_g,Subject
0,2024-11-19 23:09:33.023000+00:00,-0.504886,9.611969,-1.824527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2024-11-19 23:09:33.034000+00:00,-0.519243,9.657433,-1.869991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2024-11-19 23:09:33.032000+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122334.0,168615.0,204001.0,180128.0,1
3,2024-11-19 23:09:33.037000+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122334.0,168615.0,204001.0,180128.0,1
4,2024-11-19 23:09:33.022000+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122334.0,168615.0,204001.0,180128.0,1


# Masimo data from 11/12

In [90]:
import pandas as pd

# Step 1: Clean the raw SpO2 sensor data

# Load the CSV into a DataFrame
input_csv_path = "/Users/sjtok/bc_infection/wearable-data-science/data/11_12_2024/SPO2/0_Seijung Kim_11_14_24.csv"
df = pd.read_csv(input_csv_path)

# Remove empty rows
df.dropna(how='all', inplace=True)

# Drop rows that are exactly the same as the header (if they appear in the middle of the file)
df = df[df.iloc[:, 0] != "Session"]

# Define the column headers
column_headers = ["Session", "Index", "Timestamp", "Date", "Time", "O2 Saturation", "Pulse Rate", "Perfusion Index"]
df.columns = column_headers

# Step 2: Convert the 'Date' and 'Time' columns to a 'Timestamp' column in UTC format

# Remove the timezone part from the "Time" column (e.g., "Eastern Standard Time")
df["Time"] = df["Time"].str.replace(r' \w+ \w+ Time$', '', regex=True)

# Combine 'Date' and 'Time' to form a new 'Timestamp' column
df["Timestamp"] = df["Date"] + " " + df["Time"]

# Convert the 'Timestamp' column to datetime with the expected format
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%A, %B %d, %Y %I:%M:%S %p", errors='coerce')

# Check if any timestamps failed to parse
if df["Timestamp"].isna().any():
    print(f"Some Timestamps could not be parsed: {df[df['Timestamp'].isna()]}")

# Convert 'Timestamp' to 'America/New_York' timezone, then to UTC
df["Timestamp"] = df["Timestamp"].dt.tz_localize("America/New_York", ambiguous='NaT')
df["Timestamp"] = df["Timestamp"].dt.tz_convert("UTC")

# Step 3: Load the activity timestamps for splitting the data

activity_timestamps_path = "/Users/sjtok/bc_infection/wearable-data-science/data/11_12_2024/timestamps.csv"
activity_df = pd.read_csv(activity_timestamps_path)

# Convert 'Start Time' and 'End Time' to datetime
activity_df['Start Time'] = pd.to_datetime(activity_df['Start Time'])
activity_df['End Time'] = pd.to_datetime(activity_df['End Time'])

# Step 4: Split the data into different activities

# Iterate over the activities and filter the SpO2 data
for _, row in activity_df.iterrows():
    activity = row['Activity']
    start_time = row['Start Time']
    end_time = row['End Time']
    
    # Filter the main DataFrame based on the timestamp range
    activity_data = df[(df['Timestamp'] >= start_time) & (df['Timestamp'] <= end_time)]
    
    # Save the filtered data for this activity to a new CSV file
    output_path = f"/Users/sjtok/bc_infection/wearable-data-science/data/11_12_2024/Spliced/SPO2_{activity}.csv"
    activity_data.to_csv(output_path, index=False)
    
    print(f"Filtered data for '{activity}' saved at {output_path}")


# return df


TypeError: Invalid comparison between dtype=datetime64[ns, UTC] and Timestamp